In [6]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image


# Create folder
os.makedirs("plots", exist_ok=True)

# seedsfor reproduction
np.random.seed(123)
torch.manual_seed(123)

# choose device, cuda or cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

#
# make gif out of pngss

def save_gif_PIL(outfile, files, fps=15, loop=0):
    imgs = [Image.open(f) for f in files]
    imgs[0].save(outfile, save_all=True, append_images=imgs[1:],
                 duration=int(1000/fps), loop=loop)


# Fully connected network (FCN)

class FCN(nn.Module):
    def __init__(self, N_INPUT, N_OUTPUT, N_HIDDEN, N_LAYERS):
        super().__init__()
        activation = nn.Tanh
        self.fcs = nn.Sequential(
            nn.Linear(N_INPUT, N_HIDDEN),
            activation()
        )
        self.fch = nn.Sequential(
            *[nn.Sequential(nn.Linear(N_HIDDEN, N_HIDDEN), activation())
              for _ in range(N_LAYERS-1)]
        )
        self.fce = nn.Linear(N_HIDDEN, N_OUTPUT)

    def forward(self, x):
        x = self.fcs(x)
        x = self.fch(x)
        x = self.fce(x)
        return x


# Load dataset

all_data = np.load("pendulum_datasets_with_noise.npy", allow_pickle=True)
sample = all_data[6]

params = sample["params"]
print("Loaded parameters:", params)

t_dense = sample["t_dense"]
theta_dense = sample["theta_dense"]
omega_dense = sample["omega_dense"]
x_dense = sample["x_dense"]
y_dense = sample["y_dense"]

t_obs = sample["t_obs"]
x_obs = sample["x_obs"]
y_obs = sample["y_obs"]

# choose number of datapoints, try extra polation
N = 10
t_obs = t_obs[:N]
x_obs = x_obs[:N]
y_obs = y_obs[:N]

g = float(params["g"])
L = float(params["L"])
b = float(params["b"])


# NORMALIZATION, apparently performs PINN better or best for inputs (-1,1)

t_min, t_max = t_dense.min(), t_dense.max()
print(f"t_min, t_max : {t_min}, {t_max}")

def norm_t(t):
    return 2*(t - t_min)/(t_max - t_min) - 1

def denorm_t(tn):
    return (tn + 1)/2 * (t_max - t_min) + t_min

# scaling factor for derivatives 
scale = 2.0 / (t_max - t_min)
print(f"scale: {scale}")

t_dense_n = norm_t(t_dense)
t_obs_n   = norm_t(t_obs)

# convert to torch
t_dense_t = torch.tensor(t_dense_n, dtype=torch.float32, device=device).view(-1,1)
t_obs_t   = torch.tensor(t_obs_n,   dtype=torch.float32, device=device).view(-1,1)

x_obs_t = torch.tensor(x_obs, dtype=torch.float32, device=device).view(-1,1)
y_obs_t = torch.tensor(y_obs, dtype=torch.float32, device=device).view(-1,1)

theta_dense_t = torch.tensor(theta_dense, dtype=torch.float32, device=device).view(-1,1)

theta_obs = np.arctan2(x_obs, -y_obs)


# Plot

def plot_result(t_dense, theta_true, t_obs_n, theta_data, theta_pred, step, title, fname):
    plt.figure(figsize=(8,4))
    plt.plot(t_dense, theta_true, color="tab:green", linewidth=2, alpha=0.8, label="Exact θ(t)")
    w = np.sqrt(g/L)
    # small angle  approx
    # small = np.sin(w * (denorm_t(t_dense) - denorm_t(t_dense)[0]))
    # plt.plot(t_dense, small, color="tab:gray", ls='--', linewidth=2, alpha=0.6, label="Small-angle approx")

    # pred
    plt.plot(t_dense, theta_pred, color="tab:blue", linewidth=3, alpha=0.85, label="Network prediction")


    t_obs_denorm = denorm_t(t_obs_n)
    plt.scatter(norm_t(t_obs_denorm), theta_data, s=60, color="tab:orange",
                alpha=0.7, label="Training data")

    plt.text(1.05, 0.7, f"step: {step}", transform=plt.gca().transAxes)
    l = plt.legend(loc=(1.01,0.15), frameon=False)
    plt.setp(l.get_texts(), color="k")
    plt.xlim(-1.05,1.05)
    plt.ylim(-2,2)
    plt.axis("off")
    plt.savefig(fname, dpi=120, bbox_inches='tight', pad_inches=0.1)
    plt.close()


# NN, no phy

print("\nTraining NN...")
model = FCN(1,1,32,3).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

files_nn = []

for i in range(5000):
    opt.zero_grad()
    theta_pred = model(t_obs_t)

    x_pred = L * torch.sin(theta_pred)
    y_pred = -L * torch.cos(theta_pred)

    loss = torch.mean((x_pred - x_obs_t)**2 + (y_pred - y_obs_t)**2)
    loss.backward()
    opt.step()

    if (i+1) % 10 == 0:
        with torch.no_grad():
            th_full = model(t_dense_t).cpu().numpy().flatten()

        fname = f"plots/nn_{i+1:05d}.png"
        plot_result(
            t_dense_n, theta_dense, t_obs_n, theta_obs, th_full,
            i+1, "NN", fname
        )
        files_nn.append(fname)

save_gif_PIL("nn.gif", files_nn, fps=15)
print("Saved nn.gif")


# PINN 

print("\nTraining PINN...")
model = FCN(1,1,32,3).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-4)

t_col = np.linspace(t_dense.min(), t_dense.max(), 200)
t_col_n = torch.tensor(norm_t(t_col), dtype=torch.float32, device=device).view(-1,1)
t_col_n.requires_grad_(True)

files_pinn = []

for i in range(20000):
    opt.zero_grad()

    # data loss
    theta_d = model(t_obs_t)
    x_pred = L * torch.sin(theta_d)
    y_pred = -L * torch.cos(theta_d)
    loss_data = torch.mean((x_pred - x_obs_t)**2 + (y_pred - y_obs_t)**2)

    # physics loss
    theta_col = model(t_col_n)

    dtheta_dt_n = torch.autograd.grad(theta_col, t_col_n,
                                      torch.ones_like(theta_col),
                                      create_graph=True)[0]
    d2theta_dt2_n = torch.autograd.grad(dtheta_dt_n, t_col_n,
                                        torch.ones_like(dtheta_dt_n),
                                        create_graph=True)[0]

    # scale derivatives 
    dtheta_dt = scale * dtheta_dt_n
    d2theta_dt2 = scale**2 * d2theta_dt2_n

    #  physics ODE
    phys = d2theta_dt2 + (b/L) * dtheta_dt + (g/L) * torch.sin(theta_col)

    loss_phys = torch.mean(phys**2)

    # weight phys loss, high phys loss might enforce 0 function as best solution
    loss = loss_data + 0.01 * loss_phys
    loss.backward()
    opt.step()

    if (i+1) % 150 == 0:
        with torch.no_grad():
            th_full = model(t_dense_t).cpu().numpy().flatten()

        fname = f"plots/pinn_{i+1:05d}.png"
        plot_result(
            t_dense_n, theta_dense, t_obs_n, theta_obs, th_full,
            i+1, "PINN", fname
        )
        files_pinn.append(fname)

save_gif_PIL("pinn.gif", files_pinn, fps=15)
print("Saved pinn.gif")
print("Done.")


Device: cpu
Loaded parameters: {'g': 9.81, 'L': 0.8, 'b': 0.0, 'theta0': 0.5, 'omega0': 0.2, 'noise_sigma': 0.02}
t_min, t_max : 0.0, 9.999999999999831
scale: 0.20000000000000337

Training NN...
Saved nn.gif

Training PINN...
Saved pinn.gif
Done.
